# Example 03: Regional top non-Hermitian retarded bath in a two-terminal square lattice

This notebook demonstrates how to **inject a regional non-Hermitian retarded self-energy** on the top edge of a two-terminal device using the experimental/regional RHS path already present in TDNEGF.

The objective is to analyze the resulting **effective non-Hermitian Hamiltonian** and the associated spectral and spatial signatures (including boundary accumulation / skin-effect-like diagnostics), while keeping standard left/right leads unchanged.

> Scope note: this example currently adds a **retarded-only** top bath contribution. It is not yet a full lesser/greater bath treatment.

## Imports

In [ ]:
using Pkg
Pkg.activate(joinpath(dirname(dirname(@__DIR__))))

using TDNEGF
using DifferentialEquations
using LinearAlgebra
using Statistics
using PyPlot

println("Number of Julia threads: ", Threads.nthreads())
println("Number of BLAS threads: ", BLAS.get_num_threads())

## User-editable parameters

In [ ]:
# Toggle between a quick run (default) and a larger run.
quick_test = true

if quick_test
    Nx, Ny = 4, 3
    t_end = 25.0
    dt_save = 0.5
    sweep_gammay = [0.0, 0.05, 0.10, 0.15]
else
    Nx, Ny = 8, 4
    t_end = 50.0
    dt_save = 0.25
    sweep_gammay = collect(range(0.0, 0.20; length=6))
end

# Electronic model parameters
Nσ = 2
N_orb = 1
γ = 1.0
γso = 0.50 + 0.0im
β = 33.0
N_λ1 = 49
N_λ2 = 20

# Static onsite terms in the device Hamiltonian (for interpretation and sweeps)
μ = 0.0
Bz = 0.15

# Top bath parameters (Case A and Case B below)
gamma0 = 0.20
gammay_caseA = 0.0
gammay_caseB = 0.10

reltol = 1e-6
abstol = 1e-8

tspan = (0.0, t_end)

println("quick_test = ", quick_test)
println("Lattice: Nx=$(Nx), Ny=$(Ny), N_sites=$(Nx*Ny)")
println("Bath params: gamma0=$(gamma0), gammay(A)=$(gammay_caseA), gammay(B)=$(gammay_caseB)")

## Geometry and site indexing

In [ ]:
@inline local_index(x::Int, y::Int, Ny::Int) = (x - 1) * Ny + y

@inline function local_subrange(site::Int, N_loc::Int)
    a = (site - 1) * N_loc + 1
    b = site * N_loc
    return a:b
end

function top_edge_coupled_indices(; Nx::Int, Ny::Int, N_loc::Int)
    top_sites = [local_index(x, Ny, Ny) for x in 1:Nx]
    idx_couple = Int[]
    for s in top_sites
        append!(idx_couple, collect(local_subrange(s, N_loc)))
    end
    return top_sites, idx_couple
end

function site_scalar_to_grid(values::AbstractVector, Nx::Int, Ny::Int)
    @assert length(values) == Nx * Ny
    # Row 1 of the heatmap corresponds to the physical top edge (y = Ny)
    [values[local_index(x, y, Ny)] for y in Ny:-1:1, x in 1:Nx]
end

## Device Hamiltonian

In [ ]:
function add_uniform_onsite!(H::AbstractMatrix{ComplexF64}; Nx::Int, Ny::Int, N_loc::Int, μ::Float64, Bz::Float64)
    σz = ComplexF64[1 0; 0 -1]
    onsite = (-μ) * Matrix{ComplexF64}(I, N_loc, N_loc) + Bz * σz
    for site in 1:(Nx * Ny)
        r = local_subrange(site, N_loc)
        H[r, r] .+= onsite
    end
    return H
end

## Standard left/right leads

In [ ]:
function build_two_terminal_blocks(p_model::ModelParamsTDNEGF; β::Float64, N_λ1::Int, N_λ2::Int)
    Rλ, zλ = load_poles_square(N_λ1, N_λ2)

    Σᴸ_nλ = build_Σᴸ_nλ(Rλ, zλ, p_model.Ny, p_model.Nσ, p_model.N_orb,
                       p_model.N_λ1, p_model.N_λ2; β=β, γ=1.0)
    Σᴳ_nλ = build_Σᴳ_nλ(Rλ, zλ, p_model.Ny, p_model.Nσ, p_model.N_orb,
                       p_model.N_λ1, p_model.N_λ2; β=β, γ=1.0)
    χ_nλ = build_χ_nλ(zλ, p_model.Ny, p_model.Nσ, p_model.N_orb,
                      p_model.N_λ1, p_model.N_λ2; β=β, γ=1.0)

    ξ_anR = build_ξ_an(p_model.Nx, p_model.Ny, p_model.Nσ, p_model.N_orb; xcol=p_model.Nx, y_coup=1:p_model.Ny)
    ξ_anL = build_ξ_an(p_model.Nx, p_model.Ny, p_model.Nσ, p_model.N_orb; xcol=1,          y_coup=1:p_model.Ny)

    left_block = SelfEnergyBlock(:left, p_model.Nc, p_model.N_λ1, p_model.N_λ2, Σᴸ_nλ, Σᴳ_nλ, χ_nλ, ξ_anL)
    right_block = SelfEnergyBlock(:right, p_model.Nc, p_model.N_λ1, p_model.N_λ2, Σᴸ_nλ, Σᴳ_nλ, χ_nλ, ξ_anR)
    return [left_block, right_block]
end

## Top non-Hermitian retarded bath

In [ ]:
function build_top_bath_ΣR(; n_coupled_sites::Int, N_loc::Int, gamma0::Float64, gammay::Float64)
    @assert N_loc == 2 "This example assumes one orbital and spin-1/2 (N_loc=2)."
    @assert gamma0 >= 0.0 "gamma0 should be non-negative."
    if gamma0 < abs(gammay)
        @warn "gamma0 < |gammay|: one spin channel can become amplifying (effective gain)."
    end

    σy = ComplexF64[0 -im; im 0]
    Σ_site = -1im .* (gamma0 .* Matrix{ComplexF64}(I, N_loc, N_loc) .+ gammay .* σy)
    ΣR_loc = kron(Matrix{ComplexF64}(I, n_coupled_sites, n_coupled_sites), Σ_site)
    return Σ_site, ΣR_loc
end

function embed_local_block(idx_couple::Vector{Int}, block_local::AbstractMatrix{ComplexF64}, Nc::Int)
    @assert size(block_local, 1) == size(block_local, 2) "block_local must be square."
    @assert maximum(idx_couple) <= Nc "idx_couple contains indices larger than Nc."
    @assert size(block_local, 1) == length(idx_couple) "Dimension mismatch: block_local is $(size(block_local)), but length(idx_couple)=$(length(idx_couple))."

    Σ_global = zeros(ComplexF64, Nc, Nc)
    Σ_global[idx_couple, idx_couple] .= block_local
    return Σ_global
end

function make_case(; Nx::Int, Ny::Int, Nσ::Int, N_orb::Int,
                   γ::Float64, γso, β::Float64, N_λ1::Int, N_λ2::Int,
                   μ::Float64, Bz::Float64,
                   gamma0::Float64, gammay::Float64)

    p_model = ModelParamsTDNEGF(Nx=Nx, Ny=Ny, Nσ=Nσ, N_orb=N_orb, Nα=2, N_λ1=N_λ1, N_λ2=N_λ2)

    H_device = build_H_ab(; Nx, Ny, Nσ, N_orb, γ=γ, γso=γso)
    add_uniform_onsite!(H_device; Nx=Nx, Ny=Ny, N_loc=p_model.N_loc, μ=μ, Bz=Bz)

    p_model.H_ab .= H_device
    p_model.H0_ab .= H_device

    blocks = build_two_terminal_blocks(p_model; β=β, N_λ1=N_λ1, N_λ2=N_λ2)
    Δ_blocks = ComplexF64[0.5 + 0.0im, -0.5 + 0.0im]

    top_sites, idx_couple = top_edge_coupled_indices(; Nx=Nx, Ny=Ny, N_loc=p_model.N_loc)
    Σ_site, ΣR_loc = build_top_bath_ΣR(; n_coupled_sites=length(top_sites), N_loc=p_model.N_loc, gamma0=gamma0, gammay=gammay)
    ΣR_top_global = embed_local_block(idx_couple, ΣR_loc, p_model.Nc)

    top_bath = TDNEGF.ExtraBathSpec(:nonhermitian_retarded, :top_edge_nonhermitian, idx_couple, ΣR_loc, true)
    p_blocks = ExperimentalBlockRHSParams(p_model.H_ab, blocks, Δ_blocks, p_model, [top_bath])

    u0 = zeros(ComplexF64, p_blocks.dims_ρ_ab[1]^2 + p_blocks.aux_layout.total_size)

    println("--- Top-bath sanity checks (gammay=$(gammay)) ---")
    println("top row sites        : ", top_sites)
    println("idx_couple           : ", idx_couple)
    println("length(idx_couple)   : ", length(idx_couple))
    println("size(Σ_site)         : ", size(Σ_site))
    println("size(ΣR_loc)         : ", size(ΣR_loc))
    println("size(ΣR_top_global)  : ", size(ΣR_top_global))
    println("Consistency local    : ", size(ΣR_loc, 1) == length(idx_couple))
    println("Affected sites       : ", top_sites)

    return (; p_model, p_blocks, u0, top_sites, idx_couple, Σ_site, ΣR_loc, ΣR_top_global, gammay)
end

## Solver / run

In [ ]:
function run_case(case_setup; tspan=(0.0, 25.0), reltol=1e-6, abstol=1e-8, dt_save=0.5)
    p_model, p_blocks, u0 = case_setup.p_model, case_setup.p_blocks, case_setup.u0

    save_times = collect(range(tspan[1], tspan[2]; step=dt_save))
    prob = ODEProblem(eom_tdnegf_blocks!, u0, tspan, p_blocks)
    sol = solve(prob, Vern7(); adaptive=true, saveat=save_times, dense=false, reltol=reltol, abstol=abstol)

    obs = ObservablesTDNEGF(p_model; N_tmax=length(sol.t), N_leads=length(p_blocks.blocks))
    obs.t = sol.t

    for (it, ut) in enumerate(sol.u)
        obs.idx = it
        ptr = pointer_blocks(ut, p_blocks.dims_ρ_ab, p_blocks.aux_layout)
        obs_n_i!(ptr, p_model, obs)
        obs_σ_i!(ptr, p_model, obs)
        obs_Ixα!(ptr, p_blocks, obs)
    end

    return sol, obs
end

caseA = make_case(; Nx, Ny, Nσ, N_orb, γ, γso, β, N_λ1, N_λ2, μ, Bz, gamma0, gammay=gammay_caseA)
caseB = make_case(; Nx, Ny, Nσ, N_orb, γ, γso, β, N_λ1, N_λ2, μ, Bz, gamma0, gammay=gammay_caseB)

solA, obsA = run_case(caseA; tspan=tspan, reltol=reltol, abstol=abstol, dt_save=dt_save)
solB, obsB = run_case(caseB; tspan=tspan, reltol=reltol, abstol=abstol, dt_save=dt_save)

println("Case A completed: gammay = ", caseA.gammay, ", N_t = ", length(obsA.t))
println("Case B completed: gammay = ", caseB.gammay, ", N_t = ", length(obsB.t))

## Effective Hamiltonian diagnostics

In [ ]:
function heff(case_setup)
    Matrix(case_setup.p_model.H_ab) + case_setup.ΣR_top_global
end

function site_weights_from_state(vec_state::AbstractVector, Nx::Int, Ny::Int, N_loc::Int)
    N_sites = Nx * Ny
    weights = zeros(Float64, N_sites)
    for site in 1:N_sites
        r = local_subrange(site, N_loc)
        weights[site] = real(sum(abs2, vec_state[r]))
    end
    s = sum(weights)
    return s > 0 ? weights ./ s : weights
end

function plot_heff_spectrum_and_modes(case_setup; n_modes::Int=4)
    H_eff = heff(case_setup)
    ev = eigen(H_eff)
    λ = ev.values

    fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))

    axs[1].scatter(real.(λ), imag.(λ), s=24, alpha=0.75)
    axs[1].set_xlabel("Re(E)")
    axs[1].set_ylabel("Im(E)")
    axs[1].set_title("Complex spectrum of Heff (gammay=$(case_setup.gammay))")
    axs[1].grid(alpha=0.3)

    order = sortperm(imag.(λ))
    nvals = length(order)
    nedge = min(2, nvals)
    selected = unique(vcat(order[1:nedge], order[max(1, nvals - nedge + 1):nvals]))
    selected = selected[1:min(n_modes, length(selected))]

    for (k, idx) in enumerate(selected)
        w = site_weights_from_state(ev.vectors[:, idx], case_setup.p_model.Nx, case_setup.p_model.Ny, case_setup.p_model.N_loc)
        grid = site_scalar_to_grid(w, case_setup.p_model.Nx, case_setup.p_model.Ny)
        axs[2].plot(vec(grid), label="mode $(idx), ImE=$(round(imag(λ[idx]), digits=3))", alpha=0.8)
    end
    axs[2].set_xlabel("flattened lattice index (top→bottom)")
    axs[2].set_ylabel("normalized site weight")
    axs[2].set_title("Representative right-eigenvector weights")
    axs[2].legend(fontsize=8, frameon=false)
    axs[2].grid(alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Also show a heatmap for the most amplified / least damped mode.
    idx_max_im = argmax(imag.(λ))
    w_top = site_weights_from_state(ev.vectors[:, idx_max_im], case_setup.p_model.Nx, case_setup.p_model.Ny, case_setup.p_model.N_loc)
    fig2, ax2 = plt.subplots(1, 1, figsize=(5.2, 4.2))
    im = ax2.imshow(site_scalar_to_grid(w_top, case_setup.p_model.Nx, case_setup.p_model.Ny), cmap="magma", aspect="auto")
    ax2.set_title("Spatial weight, max Im(E) mode\n(gammay=$(case_setup.gammay))")
    ax2.set_xlabel("x")
    ax2.set_ylabel("y (top at top)")
    plt.colorbar(im, ax=ax2, label="|ψ|² (normalized)")
    plt.tight_layout()
    plt.show()

    return λ
end

λA = plot_heff_spectrum_and_modes(caseA)
λB = plot_heff_spectrum_and_modes(caseB)

## Spin-density and spin-current diagnostics

In [ ]:
function spin_component_map(obs::ObservablesTDNEGF, comp::Int, t_idx::Int, Nx::Int, Ny::Int)
    vals = vec(obs.σx_i[:, comp, t_idx])
    return site_scalar_to_grid(vals, Nx, Ny)
end

function edge_contrast(vals::AbstractVector, Nx::Int, Ny::Int)
    top_idx = [local_index(x, Ny, Ny) for x in 1:Nx]
    bot_idx = [local_index(x, 1, Ny) for x in 1:Nx]
    return mean(vals[top_idx]) - mean(vals[bot_idx])
end

function plot_spin_density_comparison(obsA, obsB, caseA, caseB)
    tA = size(obsA.σx_i, 3)
    tB = size(obsB.σx_i, 3)

    fig, axs = plt.subplots(2, 3, figsize=(13, 7), constrained_layout=true)
    cm = "coolwarm"
    comps = [1, 2, 3]
    labels = ["s_x", "s_y", "s_z"]

    for (j, c) in enumerate(comps)
        mapA = spin_component_map(obsA, c, tA, caseA.p_model.Nx, caseA.p_model.Ny)
        mapB = spin_component_map(obsB, c, tB, caseB.p_model.Nx, caseB.p_model.Ny)
        vmax = max(maximum(abs.(mapA)), maximum(abs.(mapB))) + 1e-12

        imA = axs[1, j].imshow(mapA, cmap=cm, vmin=-vmax, vmax=vmax, aspect="auto")
        axs[1, j].set_title("Case A: $(labels[j]) @ t=$(round(obsA.t[end],digits=2))")
        axs[1, j].set_xlabel("x")
        axs[1, j].set_ylabel("y")

        imB = axs[2, j].imshow(mapB, cmap=cm, vmin=-vmax, vmax=vmax, aspect="auto")
        axs[2, j].set_title("Case B: $(labels[j]) @ t=$(round(obsB.t[end],digits=2))")
        axs[2, j].set_xlabel("x")
        axs[2, j].set_ylabel("y")

        plt.colorbar(imB, ax=[axs[1, j], axs[2, j]], shrink=0.75)
    end

    plt.show()

    szA = vec(obsA.σx_i[:, 3, end])
    szB = vec(obsB.σx_i[:, 3, end])
    println("Top-bottom contrast Δsz (Case A): ", edge_contrast(szA, caseA.p_model.Nx, caseA.p_model.Ny))
    println("Top-bottom contrast Δsz (Case B): ", edge_contrast(szB, caseB.p_model.Nx, caseB.p_model.Ny))
end

function plot_spin_current_comparison(obsA, obsB)
    fig, axs = plt.subplots(1, 2, figsize=(12, 4.2), sharey=true)
    comps = [1, 2, 3]
    clabel = ["x", "y", "z"]

    for c in comps
        axs[1].plot(obsA.t, obsA.Iαx[1, c, :], label="L, S$(clabel[c])")
        axs[1].plot(obsA.t, obsA.Iαx[2, c, :], "--", label="R, S$(clabel[c])")

        axs[2].plot(obsB.t, obsB.Iαx[1, c, :], label="L, S$(clabel[c])")
        axs[2].plot(obsB.t, obsB.Iαx[2, c, :], "--", label="R, S$(clabel[c])")
    end

    axs[1].set_title("Case A (gammay = 0)")
    axs[2].set_title("Case B (gammay = $(caseB.gammay))")
    for ax in axs
        ax.set_xlabel("t")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8, frameon=false, ncol=2)
    end
    axs[1].set_ylabel("Lead spin current (arb. units)")

    plt.tight_layout()
    plt.show()
end

plot_spin_density_comparison(obsA, obsB, caseA, caseB)
plot_spin_current_comparison(obsA, obsB)

## Visualization

In [ ]:
# (1) Top-edge mean density comparison
function top_edge_density_trace(obs, top_sites)
    n_top = zeros(length(obs.t))
    for s in top_sites
        n_top .+= vec(obs.n_i[s, :])
    end
    n_top ./= length(top_sites)
    return n_top
end

n_top_A = top_edge_density_trace(obsA, caseA.top_sites)
n_top_B = top_edge_density_trace(obsB, caseB.top_sites)

fig, axs = plt.subplots(1, 2, figsize=(12, 4.2))
axs[1].plot(obsA.t, n_top_A, label="Case A: gammay=0")
axs[1].plot(obsB.t, n_top_B, "--", label="Case B: gammay=$(caseB.gammay)")
axs[1].set_xlabel("t")
axs[1].set_ylabel("mean top-edge density")
axs[1].set_title("Top-edge density trace")
axs[1].grid(alpha=0.3)
axs[1].legend(frameon=false)

# (2) Parameter sweep over gammay with a boundary-accumulation diagnostic.
function run_gammay_sweep(gammay_values)
    contrasts = Float64[]
    for gy in gammay_values
        case = make_case(; Nx, Ny, Nσ, N_orb, γ, γso, β, N_λ1, N_λ2, μ, Bz, gamma0, gammay=gy)
        _, obs = run_case(case; tspan=tspan, reltol=reltol, abstol=abstol, dt_save=dt_save)
        sz_final = vec(obs.σx_i[:, 3, end])
        push!(contrasts, edge_contrast(sz_final, Nx, Ny))
    end
    return contrasts
end

contrast_sweep = run_gammay_sweep(sweep_gammay)
axs[2].plot(sweep_gammay, contrast_sweep, marker="o")
axs[2].axhline(0.0, color="k", lw=1, alpha=0.5)
axs[2].set_xlabel("gammay")
axs[2].set_ylabel("Δsz = <sz>top - <sz>bottom (final time)")
axs[2].set_title("Lightweight sweep: spin accumulation asymmetry")
axs[2].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Physical interpretation / conclusions

- **Case A (scalar damping, `gammay = 0`)** gives a baseline where the top bath acts as spin-independent loss on the top boundary.
- **Case B (spin-selective damping, `gammay ≠ 0`)** modifies both the complex spectrum of `Heff` and the spatial/spin observables, often generating stronger component-selective boundary asymmetries.
- The effective non-Hermitian Hamiltonian diagnostics (complex eigenvalues and right-eigenvector weight maps) provide a direct spectral/spatial view of how the regional retarded bath reshapes the device modes.
- Spin-density maps (`s_x,s_y,s_z`) and lead spin currents complement this by showing dynamical responses and boundary accumulation trends relevant to skin-effect-like behavior.
- **Out of scope in this notebook**: a full non-Hermitian bath treatment with consistent lesser/greater components and fluctuation-dissipation structure. Here we intentionally keep the regional extension retarded-only.